In [1]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('Titanic-Dataset.csv')

# Basic info
print("Shape of dataset:", df.shape)
print("\nColumn names and data types:")
print(df.dtypes)
print("\nFirst 5 rows:")
df.head()

Shape of dataset: (891, 12)

Column names and data types:
PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object

First 5 rows:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [2]:
# ===== DATA QUALITY REPORT =====

print("="*50)
print("DATA QUALITY REPORT")
print("="*50)

# 1. Null values per column
print("\n--- Null Values Per Column ---")
print(df.isnull().sum())

print("\n--- Percentage of Nulls ---")
print((df.isnull().sum() / len(df) * 100).round(2))

# 2. Duplicate rows
print("\n--- Duplicate Rows ---")
print("Number of duplicate rows:", df.duplicated().sum())

# 3. Data type issues
print("\n--- Data Types ---")
print(df.dtypes)

# 4. Basic statistics (helps spot range anomalies)
print("\n--- Descriptive Statistics (Numeric Columns) ---")
print(df.describe())

# 5. Unique values in categorical columns (helps spot inconsistent formatting)
print("\n--- Unique Values in Object Columns ---")
for col in df.select_dtypes(include='object').columns:
    print(f"\n{col}: {df[col].nunique()} unique values")
    print(df[col].unique()[:10])  # show first 10 unique values

DATA QUALITY REPORT

--- Null Values Per Column ---
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

--- Percentage of Nulls ---
PassengerId     0.00
Survived        0.00
Pclass          0.00
Name            0.00
Sex             0.00
Age            19.87
SibSp           0.00
Parch           0.00
Ticket          0.00
Fare            0.00
Cabin          77.10
Embarked        0.22
dtype: float64

--- Duplicate Rows ---
Number of duplicate rows: 0

--- Data Types ---
PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object

--- Descriptive Statistics (Numeric Columns) ---

In [3]:
# ===== MISSING DATA HANDLING =====

# 1. Age (19.87% missing) -> Median imputation
# Reason: Age has some outliers (very old/young passengers), median is more robust than mean
df['Age'] = df['Age'].fillna(df['Age'].median())

# 2. Cabin (77% missing) -> Drop the column
# Reason: Too many missing values to reliably impute; imputing would introduce more noise than signal
df.drop('Cabin', axis=1, inplace=True)

# 3. Embarked (0.2% missing) -> Mode imputation
# Reason: Only 2 missing values, categorical column, mode is the safest simple fill
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Verify no more nulls (except where intentionally dropped)
print("Nulls after handling:")
print(df.isnull().sum())

Nulls after handling:
PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64


In [4]:
# ===== DUPLICATE REMOVAL =====

print("Number of duplicate rows before removal:", df.duplicated().sum())

# Remove duplicates
df = df.drop_duplicates()

print("Number of duplicate rows after removal:", df.duplicated().sum())
print("New shape of dataset:", df.shape)

Number of duplicate rows before removal: 0
Number of duplicate rows after removal: 0
New shape of dataset: (891, 11)


In [5]:
# ===== STANDARDISATION =====

print("--- Checking 'Sex' column for inconsistent formatting ---")
print(df['Sex'].unique())

print("\n--- Checking 'Embarked' column for inconsistent formatting ---")
print(df['Embarked'].unique())

print("\n--- Checking 'Pclass' column ---")
print(df['Pclass'].unique())

# Standardize Sex column to consistent casing (safety measure)
df['Sex'] = df['Sex'].str.lower().str.strip()

# Standardize Embarked column (safety measure)
df['Embarked'] = df['Embarked'].str.upper().str.strip()

print("\n--- After standardisation ---")
print("Sex values:", df['Sex'].unique())
print("Embarked values:", df['Embarked'].unique())

--- Checking 'Sex' column for inconsistent formatting ---
['male' 'female']

--- Checking 'Embarked' column for inconsistent formatting ---
['S' 'C' 'Q']

--- Checking 'Pclass' column ---
[3 1 2]

--- After standardisation ---
Sex values: ['male' 'female']
Embarked values: ['S' 'C' 'Q']


In [6]:
# ===== OUTLIER DETECTION =====

def detect_outliers_iqr(column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    print(f"\n--- {column} ---")
    print(f"Q1: {Q1:.2f}, Q3: {Q3:.2f}, IQR: {IQR:.2f}")
    print(f"Lower bound: {lower_bound:.2f}, Upper bound: {upper_bound:.2f}")
    print(f"Number of outliers: {len(outliers)}")
    return outliers

# Check Age for outliers
age_outliers = detect_outliers_iqr('Age')

# Check Fare for outliers
fare_outliers = detect_outliers_iqr('Fare')

# Check SibSp for outliers
sibsp_outliers = detect_outliers_iqr('SibSp')

# Check Parch for outliers
parch_outliers = detect_outliers_iqr('Parch')


--- Age ---
Q1: 22.00, Q3: 35.00, IQR: 13.00
Lower bound: 2.50, Upper bound: 54.50
Number of outliers: 66

--- Fare ---
Q1: 7.91, Q3: 31.00, IQR: 23.09
Lower bound: -26.72, Upper bound: 65.63
Number of outliers: 116

--- SibSp ---
Q1: 0.00, Q3: 1.00, IQR: 1.00
Lower bound: -1.50, Upper bound: 2.50
Number of outliers: 46

--- Parch ---
Q1: 0.00, Q3: 0.00, IQR: 0.00
Lower bound: 0.00, Upper bound: 0.00
Number of outliers: 213


In [7]:
# ===== DATA TYPE CORRECTION =====

print("--- Data types before correction ---")
print(df.dtypes)

# PassengerId, Survived, Pclass should be treated as categorical/ID, not continuous numeric
# But we'll keep them as int since that's standard for this dataset
# Convert Sex and Embarked to 'category' dtype (proper categorical representation)
df['Sex'] = df['Sex'].astype('category')
df['Embarked'] = df['Embarked'].astype('category')
df['Pclass'] = df['Pclass'].astype('category')

# Ensure Age is float (it may have become object/mixed during median fill)
df['Age'] = df['Age'].astype(float)

# Ensure Fare is float
df['Fare'] = df['Fare'].astype(float)

# PassengerId as string/object since it's an identifier, not a number to calculate with
df['PassengerId'] = df['PassengerId'].astype(str)

print("\n--- Data types after correction ---")
print(df.dtypes)

--- Data types before correction ---
PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Embarked        object
dtype: object

--- Data types after correction ---
PassengerId      object
Survived          int64
Pclass         category
Name             object
Sex            category
Age             float64
SibSp             int64
Parch             int64
Ticket           object
Fare            float64
Embarked       category
dtype: object


In [8]:
# ===== BEFORE VS AFTER SUMMARY =====

# Re-load original dataset for comparison (fresh copy)
df_original = pd.read_csv('Titanic-Dataset.csv')

summary = pd.DataFrame({
    'Metric': [
        'Total Rows',
        'Total Columns',
        'Null Values (Total)',
        'Duplicate Rows',
        'Columns with Wrong Dtype (approx)'
    ],
    'Before Cleaning': [
        df_original.shape[0],
        df_original.shape[1],
        df_original.isnull().sum().sum(),
        df_original.duplicated().sum(),
        3  # Sex, Embarked, Pclass were object/int instead of category
    ],
    'After Cleaning': [
        df.shape[0],
        df.shape[1],
        df.isnull().sum().sum(),
        df.duplicated().sum(),
        0
    ]
})

print(summary.to_string(index=False))

                           Metric  Before Cleaning  After Cleaning
                       Total Rows              891             891
                    Total Columns               12              11
              Null Values (Total)              866               0
                   Duplicate Rows                0               0
Columns with Wrong Dtype (approx)                3               0


In [9]:
# ===== SAVE CLEANED DATASET =====

df.to_csv('Titanic_Cleaned.csv', index=False)
print("Cleaned dataset saved as 'Titanic_Cleaned.csv'")

# Verify
df_check = pd.read_csv('Titanic_Cleaned.csv')
print("\nShape of saved file:", df_check.shape)
df_check.head()

Cleaned dataset saved as 'Titanic_Cleaned.csv'

Shape of saved file: (891, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S
